In [3]:
!pip install -q transformers peft accelerate bitsandbytes

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!ls /content/drive/MyDrive

 adapters  'Colab Notebooks'   day3


In [6]:
!mkdir -p /content/adapters

!cp /content/drive/MyDrive/adapters/adapter_model.safetensors /content/adapters/
!cp /content/drive/MyDrive/adapters/adapter_config.json /content/adapters/
!cp /content/drive/MyDrive/adapters/tokenizer.json /content/adapters/
!cp /content/drive/MyDrive/adapters/tokenizer_config.json /content/adapters/

In [7]:
!ls /content/adapters

adapter_config.json	   tokenizer_config.json
adapter_model.safetensors  tokenizer.json


In [8]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.float16,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(base_model)

model = PeftModel.from_pretrained(model, "/content/adapters")

# MOST IMPORTANT LINE
model = model.merge_and_unload()

model.save_pretrained("/content/merged_model")
tokenizer.save_pretrained("/content/merged_model")

print("merged_model ready")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

merged_model ready


In [9]:
!ls /content/merged_model

chat_template.jinja  generation_config.json  tokenizer_config.json
config.json	     model.safetensors	     tokenizer.json


In [10]:
from transformers import BitsAndBytesConfig

bnb8 = BitsAndBytesConfig(load_in_8bit=True)

model8 = AutoModelForCausalLM.from_pretrained(
    "/content/merged_model",
    quantization_config=bnb8,
    device_map="auto"
)

model8.save_pretrained("/content/model-int8")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [11]:
bnb4 = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

model4 = AutoModelForCausalLM.from_pretrained(
    "/content/merged_model",
    quantization_config=bnb4,
    device_map="auto"
)

model4.save_pretrained("/content/model-int4")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [12]:
!git clone https://github.com/ggerganov/llama.cpp
%cd llama.cpp

!apt-get install -y cmake build-essential
!cmake -B build
!cmake --build build --config Release

Cloning into 'llama.cpp'...
remote: Enumerating objects: 87635, done.
remote: Counting objects: 100% (310/310), done.
remote: Compressing objects: 100% (196/196), done.
remote: Total 87635 (delta 216), reused 121 (delta 114), pack-reused 87325 (from 3)
Receiving objects: 100% (87635/87635), 364.37 MiB | 33.29 MiB/s, done.
Resolving deltas: 100% (63365/63365), done.
/content/llama.cpp
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
build-essential is already the newest version (12.9ubuntu3).
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI inf

In [13]:
!python3 convert_hf_to_gguf.py /content/merged_model \
  --outfile model.gguf \
  --outtype f16

INFO:hf-to-gguf:Loading model: merged_model
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.float16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> F16, shape = {5632, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.float16 --> F16, shape = {2048, 256}
INFO:hf-to-gguf:blk.0.attn_output.we

In [14]:
! ./build/bin/llama-quantize model.gguf model-q4.gguf q4_0

main: build = 8732 (057dba336)
main: built with GNU 11.4.0 for Linux x86_64
main: quantizing 'model.gguf' to 'model-q4.gguf' as Q4_0
llama_model_loader: loaded meta data with 30 key-value pairs and 201 tensors from model.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Merged_Model
llama_model_loader: - kv   3:                         general.size_label str              = 1.1B
llama_model_loader: - kv   4:                          llama.block_count u32              = 22
llama_model_loader: - kv   5:                       llama.context_length u32              = 2048
llama_model_loader: - kv   6:                     llama.emb

In [15]:
!./build/bin/llama-quantize model.gguf model-q4.gguf q4_0

main: build = 8732 (057dba336)
main: built with GNU 11.4.0 for Linux x86_64
main: quantizing 'model.gguf' to 'model-q4.gguf' as Q4_0
llama_model_loader: loaded meta data with 30 key-value pairs and 201 tensors from model.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Merged_Model
llama_model_loader: - kv   3:                         general.size_label str              = 1.1B
llama_model_loader: - kv   4:                          llama.block_count u32              = 22
llama_model_loader: - kv   5:                       llama.context_length u32              = 2048
llama_model_loader: - kv   6:                     llama.emb

In [16]:
!./build/bin/llama-cli -m model-q4.gguf -p "What is random choice?"


Loading model... |-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\ 


▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b8732-057dba336
model      : model-q4.gguf
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read <file>        add a text file
  /glob <pattern>     add text files using globbing pattern


> What is random choice?

|-\|/-\|/- In computer science, a random choice is a method of selecting a value from a set of possible choices without knowing the specific choice. It is also known as a randomized algorithm or random selection.

In many algorithms, random choice is used as a way to ensure that a specifi

In [17]:
!mkdir quantized


In [18]:
!mv /content/model-int8 quantized/
!mv /content/model-int4 quantized/
!mv /content/llama.cpp/model.gguf quantized/
!mv /content/llama.cpp/model-q4.gguf quantized/

In [19]:
!ls quantized

model.gguf  model-int4	model-int8  model-q4.gguf


In [20]:
import os

def get_size(path):
    total = 0
    for root, dirs, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return round(total/(1024**3),2)

print("INT8:", get_size("quantized/model-int8"), "GB")
print("INT4:", get_size("quantized/model-int4"), "GB")
print("GGUF:", os.path.getsize("quantized/model-q4.gguf")/(1024**3), "GB")

INT8: 1.15 GB
INT4: 0.75 GB
GGUF: 0.592997819185257 GB


In [21]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
!mkdir -p "/content/drive/MyDrive/day3/quantized"
!mkdir -p "/content/drive/MyDrive/day3/notebooks"

In [23]:
!cp -r quantized/* "/content/drive/MyDrive/day3/quantized/"